# Crear actas de reunión a partir de un archivo de audio

Descargué algunas actas de reuniones del Concejo Municipal de Denver y seleccioné una parte de una reunión para que la transcribamos. Puedes descargarla aquí:  
https://drive.google.com/file/d/1N_kpSojRR5RYzupz6nqM8hMSoEF_R7pU/view?usp=sharing

Si prefieres trabajar con los datos originales, el dataset de HuggingFace está [aquí](https://huggingface.co/datasets/huuuyeah/meetingbank) y el audio se puede descargar [aquí](https://huggingface.co/datasets/huuuyeah/MeetingBank_Audio/tree/main).

El objetivo de este proyecto es usar el audio para generar actas de reunión, incluidas las acciones pendientes.

Para este proyecto, puedes usar las actas de Denver o grabar algo propio.

## Otra vez, ten en cuenta: consejo para usar Colab

**Consejo:**

En medio de la ejecución de un Colab, puede aparecer un error como este:

> Runtime error: CUDA is required but not available for bitsandbytes. Please consider installing [...]

¡Es un mensaje de error súper engañoso! Por favor, no intentes cambiar las versiones de los paquetes...

Esto pasa porque Google cambió tu runtime de Colab, quizá porque Google Colab estaba muy ocupado. La solución es:

1. Menú Kernel >> Disconnect and delete runtime
2. Recarga el colab desde cero y ve al menú Edit >> Clear All Outputs
3. Conéctate a una nueva T4 con el botón de arriba a la derecha
4. Selecciona "View resources" en el menú de arriba a la derecha para confirmar que tienes GPU
5. Vuelve a ejecutar las celdas del colab, de arriba hacia abajo, empezando por las instalaciones de pip

Y todo debería funcionar muy bien. Si no, ¡pregúntame!

In [ ]:
!pip install -q --upgrade bitsandbytes accelerate transformers==4.57.6

In [ ]:
# imports

import os
import requests
from IPython.display import Markdown, display, update_display
from openai import OpenAI
from google.colab import drive
from huggingface_hub import login
from google.colab import userdata
from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer, BitsAndBytesConfig
import torch

In [ ]:
# Constantes

LLAMA = "meta-llama/Llama-3.2-3B-Instruct"

In [ ]:
# Nueva capacidad: conecta este Colab a tu Google Drive
# Mira justo debajo las instrucciones para obtener denver_extract.mp3
# Colócalo en tu Drive dentro de una carpeta llamada llms, y llámalo denver_extract.mp3

drive.mount("/content/drive")
audio_filename = "/content/drive/MyDrive/llms/denver_extract.mp3"

# Descargar denver_extract.mp3

Puedes usar el mismo archivo que yo, el extracto de las actas del concejo municipal de Denver, o probar con el tuyo.

Si quieres usar el mismo que yo, descarga mi extracto aquí y ponlo en tu Google Drive:  
https://drive.google.com/file/d/1N_kpSojRR5RYzupz6nqM8hMSoEF_R7pU/view?usp=sharing

In [ ]:
# Iniciar sesión en Hugging Face Hub

hf_token = userdata.get('HF_TOKEN')
login(hf_token, add_to_git_credential=True)

# Abrir el archivo

audio_file = open(audio_filename, "rb")

# PASO 1: Transcribir el audio

## Opción 1: Usar código abierto para la transcripción - Pipelines de Hugging Face

In [ ]:
from transformers import pipeline

pipe = pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-medium.en",
    dtype=torch.float16,
    device='cuda',
    return_timestamps=True
)

result = pipe(audio_filename)
transcription = result["text"]
print(transcription)

In [ ]:
open_source_transcription = transcription

## Opción 2: Usar OpenAI para la transcripción

In [ ]:
# Iniciar sesión en Gemini usando los Secrets de Colab (necesitas un secreto llamado GEMINI_API_KEY)

!pip install -q google-genai

from google import genai

AUDIO_MODEL = "gemini-2.5-flash"

gemini_api_key = userdata.get('GEMINI_API_KEY')
client = genai.Client(api_key=gemini_api_key)
uploaded_audio = client.files.upload(file=audio_filename)
response = client.models.generate_content(
    model=AUDIO_MODEL,
    contents=["Transcribe this audio verbatim, in the same language it is spoken.", uploaded_audio],
)
transcription = response.text
print(transcription)

In [ ]:
display(Markdown(open_source_transcription))
print("\n\n")
display(Markdown(transcription))

# PASO 2: Analizar y generar el informe

In [ ]:
system_message = """
You produce minutes of meetings from transcripts, with summary, key discussion points,
takeaways and action items with owners, in markdown format without code blocks.
"""

user_prompt = f"""
Below is an extract transcript of a Denver council meeting.
Please write minutes in markdown without code blocks, including:
- a summary with attendees, location and date
- discussion points
- takeaways
- action items with owners

Transcription:
{transcription}
"""

messages = [
    {"role": "system", "content": system_message},
    {"role": "user", "content": user_prompt}
  ]


In [ ]:
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"
)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(LLAMA)
tokenizer.pad_token = tokenizer.eos_token
inputs = tokenizer.apply_chat_template(messages, return_tensors="pt").to("cuda")
streamer = TextStreamer(tokenizer)
model = AutoModelForCausalLM.from_pretrained(LLAMA, device_map="auto", quantization_config=quant_config)
outputs = model.generate(inputs, max_new_tokens=2000, streamer=streamer)

In [ ]:
response = tokenizer.decode(outputs[0])

In [ ]:
display(Markdown(response))

# Contribución de un estudiante

El estudiante Emad S. creó una variante muy potente que usa `TextIteratorStreamer` para transmitir los resultados a una interfaz de Gradio y aprovecha hilos en segundo plano para mejorar el rendimiento. Lo comparto por si quieres echarle un vistazo a este trabajo tan interesante. ¡Gracias, Emad!

https://colab.research.google.com/drive/1Ja5zyniyJo5y8s1LKeCTSkB2xyDPOt6D